# 🧠 Jarvis Private GPT: Training Lab
이 노트북은 Jarvis의 인공지능 두뇌를 처음부터(from scratch) 학습시키기 위해 제작되었습니다.

**주요 기능:**
- 외부 Pretrained 가중치 0% (완전 처음부터 학습)
- 15M 내외의 소형 GPT 아키텍처
- 사용자 데이터 기반의 BPE Tokenizer 학습
- 학습 완료 후 Jarvis 호환 모델 파일 생성 (`trained_model.zip`)

---

## 1. 환경 설정 및 라이브러리 설치

In [ ]:
!pip install transformers tokenizers torch datasets

## 2. 데이터 준비 (Dataset Prep)
Jarvis 앱에서 수집한 `cleaned_dataset.jsonl`이 있다면 업로드하세요. 없다면 테스트용 샘플 데이터로 진행합니다.

In [ ]:
import json
import os

# 데이터가 없을 경우를 대비한 가상의 학습 데이터 생성 (한국어 중심)
sample_data = [
    {"text": "나는 민시후이고 15살이야. 코딩을 좋아해."},
    {"text": "자비스는 로컬에서 실행되는 개인용 AI 비서입니다."},
    {"text": "사용자의 프라이버시를 최우선으로 생각합니다."},
    {"text": "인공지능은 데이터와 알고리즘의 결합입니다."},
    {"text": "오늘 날씨는 아주 맑고 화창합니다."}
]

with open('training_data.jsonl', 'w', encoding='utf-8') as f:
    for line in sample_data:
        f.write(json.dumps(line, ensure_ascii=False) + '\n')

print("✅ 데이터 준비 완료.")

## 3. Tokenizer 학습 (Byte-Level BPE)
사용자의 단어 사용 패턴을 학습하여 효율적인 토크나이저를 생성합니다.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

# 토크나이저 학습
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(files=["training_data.jsonl"], vocab_size=52_000, min_frequency=2, special_tokens=[
    "<s>",
    "<pad>",
    "</s>",
    "<unk>",
    "<mask>",
])

# 저장
os.makedirs("model/final", exist_ok=True)
tokenizer.save_model("model/final")
print("✅ Tokenizer 학습 및 저장 완료.")

## 4. 모델 아키텍처 정의 (GPT-Tiny)
15.4M 파라미터 규모의 가벼운 모델로 설정합니다.

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=52_000,
    n_positions=256,
    n_ctx=256,
    n_embd=256,
    n_layer=8,
    n_head=8,
    activation_function="gelu_new",
)

# 모델 초기화 (가중치는 랜덤 생성됨 - Pretrained 아님!)
model = GPT2LMHeadModel(config)
print(f"✅ 모델 구축 완료. 파라미터 수: {model.num_parameters() / 1e6:.1f}M")

## 5. 학습 진행 (Training Room)
데이터가 적을 때는 빠르게 진행되며, GPU를 사용하면 훨씬 빠릅니다.

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments, PreTrainedTokenizerFast
from datasets import load_dataset

# 토크나이저 로드
fast_tokenizer = PreTrainedTokenizerFast(tokenizer_file="model/final/tokenizer.json")
fast_tokenizer.pad_token = "<pad>"
fast_tokenizer.model_max_length = 256

# 데이터셋 로드
dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

def tokenize_function(examples):
    tokens = fast_tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=fast_tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=20,       # Increased for small datasets
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    report_to="none",         # Disable W&B
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

print("🚀 학습 시작...")
trainer.train()
print("✅ 학습 종료.")

## 6. 결과 저장 및 압축
Jarvis 앱에 업로드할 최종 파일을 만듭니다.

In [ ]:
import shutil

# 모델 저장
model.save_pretrained("model/final")

# zip 생성
shutil.make_archive("trained_model", 'zip', "model/final")

print("--- 🏁 성공! ---")
print("왼쪽 파일 아이콘을 클릭하여 'trained_model.zip'을 다운로드하세요.")
print("Jarvis 설정의 Phase 2 경로에 이 파일을 압축 해제하면 학습된 지능이 적용됩니다.")